# Official data quality audit

Reproducible companion for the evidence generated by `scripts/run_data_quality_audit.py`. The notebook reads reviewed evidence tables; it does not recompute or mutate official data.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display
root = Path.cwd()
audit = root / 'data' / 'derived' / 'd004_data_quality_audit_v2'
summary = pd.read_csv(audit / 'audit_summary.csv')
display(summary.groupby(['dimension', 'status']).size().rename('checks').reset_index())

,dimension,status,checks
0,anomaly,REVIEW,1
1,completeness,PASS,1
2,consistency,PASS,1
3,consistency,REVIEW,1
4,coverage,PASS,1
5,distribution,REVIEW,1
6,keys,PASS,1
7,leakage,PASS,1
8,missingness,PASS,1
9,reconciliation,PASS,1


In [2]:
review = summary[summary.status != 'PASS']
display(review if len(review) else pd.DataFrame({'result': ['All checks passed']}))

,check_id,dimension,status,severity_if_failed,metric_value,threshold,details
5,additive_relationships,consistency,REVIEW,MEDIUM,1,0 mismatches,One known balance equation exception is retain...
8,anomalous_dates,anomaly,REVIEW,MEDIUM,67,review flags; not automatic deletions,Prior-only robust flags identify dates requiri...
9,extreme_users,distribution,REVIEW,INFO,103,review only; preserve raw values,Long-tail users are retained and listed for ro...


In [3]:
display(pd.read_csv(audit / 'additive_relationships.csv'))
display(pd.read_csv(audit / 'balance_continuity_summary.csv'))

,relationship,rows_checked,mismatch_rows,match_rate,max_abs_delta,sum_delta,status,severity_if_violated
0,direct_purchase_amt = purchase_bal_amt + purch...,2840421,0,1.0,0,0,PASS,HIGH
1,total_purchase_amt = direct_purchase_amt + sha...,2840421,0,1.0,0,0,PASS,HIGH
2,total_redeem_amt = consume_amt + transfer_amt,2840421,0,1.0,0,0,PASS,HIGH
3,transfer_amt = tftobal_amt + tftocard_amt,2840421,0,1.0,0,0,PASS,HIGH
4,consume_amt = category1..4 (null treated as st...,2840421,0,1.0,0,0,PASS,HIGH
5,tBalance = yBalance + total_purchase_amt - tot...,2840421,1,1.0,100,-100,REVIEW,MEDIUM


,continuity_scope,rows_checked,mismatch_rows,status
0,previous observed user row,2812380,0,PASS
1,previous calendar day only,2795780,0,PASS


In [4]:
display(pd.read_csv(audit / 'anomalous_dates.csv').head(30))
display(pd.read_csv(audit / 'user_concentration.csv'))

,report_date,metric,value,trailing_median_28d,trailing_mad_28d,prior_same_weekday_median_8,prior_same_weekday_mad_8,trailing_robust_score,weekday_robust_score,robust_score,is_flagged,method
0,2013-09-01,total_redeem_amt,59339949.0,26147365.5,6533054.5,7973588.0,5240869.0,3.426895,6.610762,6.610762,True,"max(prior-only 28d, prior same-weekday 8-point..."
1,2013-09-02,total_purchase_amt,140844739.0,45935173.5,7745531.5,55385171.0,13344621.5,8.264846,4.319470,8.264846,True,"max(prior-only 28d, prior same-weekday 8-point..."
2,2013-09-03,total_purchase_amt,80507880.0,47060492.5,8667548.5,49671883.0,3097133.0,2.602807,6.715435,6.715435,True,"max(prior-only 28d, prior same-weekday 8-point..."
3,2013-09-09,total_redeem_amt,45621186.0,26566461.5,6990920.5,18871815.5,2420239.5,1.838418,7.454718,7.454718,True,"max(prior-only 28d, prior same-weekday 8-point..."
4,2013-09-10,total_purchase_amt,94684143.0,45935173.5,7745531.5,50744515.5,3097133.0,4.245122,9.569131,9.569131,True,"max(prior-only 28d, prior same-weekday 8-point..."
5,2013-09-16,total_purchase_amt,161656210.0,47586984.5,9555474.0,64004308.5,21963759.0,8.051787,2.998818,8.051787,True,"max(prior-only 28d, prior same-weekday 8-point..."
6,2013-09-22,new_users,13.0,25.0,3.0,25.0,1.0,2.697963,8.093889,8.093889,True,"max(prior-only 28d, prior same-weekday 8-point..."
7,2013-10-06,new_users,37.0,25.0,4.0,25.5,1.0,2.023472,7.756644,7.756644,True,"max(prior-only 28d, prior same-weekday 8-point..."
8,2013-10-13,new_users,67.0,25.5,3.5,25.5,2.5,7.997533,11.196547,11.196547,True,"max(prior-only 28d, prior same-weekday 8-point..."
9,2013-10-14,new_users,62.0,26.5,3.5,27.0,5.0,6.841263,4.721435,6.841263,True,"max(prior-only 28d, prior same-weekday 8-point..."


,metric,group,user_count,amount,share
0,purchase_total,top_1_user,1,811225952,0.008761
1,purchase_total,top_10_users,10,4276672745,0.046189
2,purchase_total,top_0_1_percent_users,29,8206586416,0.088633
3,purchase_total,top_1_percent_users,281,30205283933,0.326222
4,purchase_total,top_5_percent_users,1403,62392935762,0.673855
5,purchase_total,top_10_percent_users,2805,78321393773,0.845885
6,redeem_total,top_1_user,1,710187572,0.009766
7,redeem_total,top_10_users,10,3774444718,0.051905
8,redeem_total,top_0_1_percent_users,29,7373829062,0.101403
9,redeem_total,top_1_percent_users,281,26685061830,0.366965


In [5]:
display(pd.read_csv(audit / 'leakage_audit.csv'))
display(pd.read_csv(audit / 'read_performance.csv'))

,scope,check,observed,required,status
0,user balance,maximum data date <= competition cutoff,2014-08-31,2014-08-31,PASS
1,fund yield,maximum data date <= competition cutoff,2014-08-31,2014-08-31,PASS
2,observed SHIBOR,maximum data date <= competition cutoff,2014-08-29,2014-08-31,PASS
3,derived SHIBOR,maximum data date <= competition cutoff,2014-08-31,2014-08-31,PASS
4,daily aggregate,maximum data date <= competition cutoff,2014-08-31,2014-08-31,PASS
5,backtest_2013_09_v1.json,explicit train/holdout chronology,train_end=2013-08-31; holdout_start=2013-09-01,train_end < holdout_start and holdout_end <= 2...,PASS
6,holdout_2014_08_v1.json,explicit train/holdout chronology,train_end=2014-08-01; holdout_start=2014-08-02,train_end < holdout_start and holdout_end <= 2...,PASS
7,rolling_30d_2014_03_08_v1.json:rolling_2014_03,training maximum implied as start-1; holdout w...,train_max=2014-02-28; holdout_end=2014-03-30,train_max < holdout_start and holdout_end <= 2...,PASS
8,rolling_30d_2014_03_08_v1.json:rolling_2014_04,training maximum implied as start-1; holdout w...,train_max=2014-03-31; holdout_end=2014-04-30,train_max < holdout_start and holdout_end <= 2...,PASS
9,rolling_30d_2014_03_08_v1.json:rolling_2014_05,training maximum implied as start-1; holdout w...,train_max=2014-04-30; holdout_end=2014-05-30,train_max < holdout_start and holdout_end <= 2...,PASS


,case,trials,median_seconds,min_seconds,max_seconds,rows
0,csv selected columns full period,3,0.874623,0.860699,0.876135,2840421
1,parquet selected columns full period,3,0.035234,0.034743,0.038653,2840421
2,csv selected columns then 2014-08 filter,3,0.934729,0.932891,0.937752,382923
3,parquet selected columns with 2014-08 pushdown,3,0.012586,0.009298,0.016177,382923
